In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
import seaborn as sns
from scipy import stats as scipystats


sns.set_style("darkgrid")
sns.set_theme(style = "darkgrid")

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 600)
pd.set_option('display.precision', 3)
pd.set_option('display.expand_frame_repr', False)

# ***DATA ANALYSIS BEFORE ACDC DOWNSTREAM NOTEBOOK WAS PUBLISHED:*** 
# ***-import concatenated acdc_output file for both reps of SCD live-cell microscopy and concatenate*** 
# ***-save this raw data file (with frame-specific information) as 'data.csv'***
# ***-for each position in each rep, calculate phase quantities for daughters and mothers*** 
# ***-assign strain names to positions for each rep***
# ***-save the new table (without frame-specific information) as separate 'analysed.csv' file*** 

In [2]:
granddata20 = pd.read_csv(r"Y:\Analysed Live cell microscopy data\20210120\TIFFs\grand_dataset_20210120.csv",
                 index_col=['Position','Cell_ID']
                 ).sort_index().reset_index()

granddata20['date'] = 20012021
# print (granddata21)


granddata21 = pd.read_csv(r"Y:\Analysed Live cell microscopy data\20210121\MIA_SCD_20210121_6strains.nd2\grand_dataset_20210121.csv",
                 index_col=['Position','Cell_ID']
                 ).sort_index().reset_index()

granddata21['date'] = 21012021


# granddata21.columns = granddata21.columns.str.replace("'"," ")
#
# granddata21 = granddata21.rename(columns={'Cell cycle stage': 'cell_cycle_stage', '# of cycles': 'generation_num',
#                                             'Relative s ID': 'relative_ID', 'Relationship': 'relationship', 'Emerg_frame_i': 'emerg_frame_i',
#                                             'Division_frame_i': 'division_frame_i', 'cell_volume_fl': 'cell_vol_fl'})
#
# granddatatotal = pd.concat([granddata21, granddata20])
granddatatotal = pd.concat([granddata20, granddata21])


print((granddatatotal['is_cell_dead']).unique())

granddatatotal.loc[(granddatatotal.is_cell_dead == '0'),'is_cell_dead']=0
granddatatotal.loc[(granddatatotal.is_cell_dead == '1'),'is_cell_dead']=1
granddatatotal.loc[(granddatatotal.is_cell_dead == 'FALSE'),'is_cell_dead']=0

print((granddatatotal['is_cell_dead']).unique())
granddatatotal.to_csv(r"Y:\test_dfs_for_final_code\CombinedTRs SCD LCM data.csv")
date_df = []

import warnings

with warnings.catch_warnings(record=True):

    for date in granddatatotal['date'].unique():
        granddata = granddatatotal[(granddatatotal['date'] == date) & (granddatatotal['is_cell_dead'] == 0)]
    
    
    
        dfs = []
        dfss =[]
        expsumm = []
        keys = []
    
        colz = ['Position_n', 'cell_cycle_num', 'Cell_ID', 'Birth frame', 'Birth vol fl', 'last_G1_frame', 'size_G1_end', 'vol_added_G1']
        colzy = ['Position_n', 'cell_cycle_num','Cell_ID', 'Div frame', 'Div vol fl', 'last_G1_frame', 'size_G1_end', 'vol_added_G1']
        cellsize = []
        cellsize_ccr = []
        s_size = []
        positions = granddata['Position'].unique()
    
    
    
        for pos in positions:
    
            keys.append(pos)
            df = granddata[granddata['Position'] == pos]
    
    
    
    
            last_frame_i = df['frame_i'].unique().max()
    
    
    
            # Compute lengths of G1 first cycle for each cell
            df_G1 = df[df['cell_cycle_stage'] == 'G1']
            df_G1_firstcycle = df_G1[df_G1['generation_num'] == 1.0]
    
    
    
    
            grouped_G1_firstcycle = df_G1_firstcycle.groupby('Cell_ID')
            cycleno = 1
    
            df_len_G1 = grouped_G1_firstcycle.size().to_frame('len_G1_cycle1')
            cells = df_G1_firstcycle['Cell_ID'].unique()
    
            for cell in cells:
                cell_df_G1_cc1 = df_G1_firstcycle[df_G1_firstcycle['Cell_ID'] == cell]
    
                div_df = cell_df_G1_cc1[(cell_df_G1_cc1['division_frame_i'] != -1)  ]
                if div_df.empty == False:
    
                    birth_frame = (div_df['division_frame_i']).min()
    
                    cell_birth_df = cell_df_G1_cc1[cell_df_G1_cc1['frame_i'] == birth_frame]
    
                    size_birth = cell_birth_df['cell_vol_fl'].min()
    
                    last_G1_frame = div_df['frame_i'].iloc[-1]
                    size_G1_end_df = div_df[div_df['frame_i'] == last_G1_frame]
                    size_G1_end = size_G1_end_df['cell_vol_fl'].iloc[0]
    
                    vol_added_G1 = size_G1_end - size_birth
    
    
                else:
    
                    birth_frame = np.nan
                    size_birth = np.nan
                    last_G1_frame = np.nan
                    size_G1_end = np.nan
                    vol_added_G1 = np.nan
                cellsizestats= [pos, cycleno, cell, birth_frame, size_birth, last_G1_frame, size_G1_end, vol_added_G1]
                cellsize.append(cellsizestats)
    
    
    
    
    
    
    
            # Get the last frame_i of G1 for each group.
            # If the last frame_i is equal to the last overall frame_i then that cell didn't finish G1
            df_last_frame_i_G1 = (
                                  grouped_G1_firstcycle['frame_i']
                                  .last()
                                  .to_frame('last_frame_i_G1_cycle1')
                                  )
    
            df_last_frame_i_G1 = df_last_frame_i_G1[df_last_frame_i_G1['last_frame_i_G1_cycle1'] == last_frame_i]
    
    
            # Put NaN on cells that did not finish G1
            IDs_not_finished_G1_cycle1 = df_last_frame_i_G1.index
    
            df_len_G1.loc[IDs_not_finished_G1_cycle1] = np.nan
    
    
    
            # Compute lengths of G1 cycles > 1 for each cell
            cycles = df_G1['generation_num'].unique()
            cycles_greater_than1 = [cycle for cycle in cycles if cycle > 1.0]
            dfs_G1_cycles_greater_than1 = []
            colnames = []
            collzz= []
    
            # Iterate all cycles > 1 and calculate length of each one of them
            for cycle in cycles_greater_than1:
                df_G1_cycle = df_G1[df_G1['generation_num'] == cycle]
                grouped_G1_cycle = df_G1_cycle.groupby('Cell_ID')
                colname = f'len_G1_cycle{cycle}'
                colnames.append(colname)
                df_len_G1_cycle = grouped_G1_cycle.size().to_frame(colname)
                # Get the last frame_i of G1 for each group.
                df_last_frame_i_G1 = (
                                      grouped_G1_cycle['frame_i']
                                      .last()
                                      .to_frame('last_frame_i')
                                      )
    
                df_last_frame_i_G1 = df_last_frame_i_G1[df_last_frame_i_G1['last_frame_i'] == last_frame_i]
                cells = df_G1_cycle['Cell_ID'].unique()
    
    
                for cell in cells:
                    cell_df_G1_ccr = df_G1_cycle[df_G1_cycle['Cell_ID'] == cell]
    
                    div_df = cell_df_G1_ccr[(cell_df_G1_ccr['division_frame_i'] != -1) ]
                    if div_df.empty == False:
    
    
    
                        div_frame = (div_df['division_frame_i']).min()
    
                        cell_div_df = cell_df_G1_ccr[cell_df_G1_ccr['frame_i'] == div_frame]
    
    
                        size_div = cell_div_df['cell_vol_fl'].min()
                        last_G1_frame = div_df['frame_i'].iloc[-1]
                        size_G1_end_df = div_df[div_df['frame_i'] == last_G1_frame]
                        size_G1_end = size_G1_end_df['cell_vol_fl'].iloc[0]
    
                        vol_added_G1 = size_G1_end - size_div
    
                    else:
    
                        div_frame = np.nan
                        size_div = np.nan
                        last_G1_frame = np.nan
                        size_G1_end = np.nan
                        vol_added_G1 = np.nan
    
                    cellsizestats1= [pos, cycle, cell, div_frame, size_div, last_G1_frame, size_G1_end, vol_added_G1]
                    cellsize_ccr.append(cellsizestats1)
                # Put NaN on cells that did not finish G1
                IDs_not_finished_G1_cycle = df_last_frame_i_G1.index
                df_len_G1_cycle.loc[IDs_not_finished_G1_cycle] = np.nan
    
                dfs_G1_cycles_greater_than1.append(df_len_G1_cycle)
    
            # Join df_len_G1 with all other cycles to create one dataframe
            df_len_G1 = df_len_G1.join(dfs_G1_cycles_greater_than1, how='outer')
    
    
    
    
            # Compute the mean length of all other cycles
            df_len_G1['mean_len_G1_allother_cycles'] = df_len_G1[colnames].mean(axis=1)
    
    
    
            # Put nan for cells in the first frame at the first cell cycle
            df_mothers_frame0 = df[(df['frame_i']==0) & (df['generation_num']== 2.0)]
    
            IDs_frame0 = df_mothers_frame0['Cell_ID'].unique()
    
    
            for xyz in IDs_frame0:
                if (xyz in df_len_G1.index) == True:
                    df_len_G1['len_G1_cycle1'].loc[xyz] = np.nan
                    df_len_G1['len_G1_cycle2'].loc[xyz] = np.nan
    
    
            df_len_G1_nomean = df_len_G1.drop(['mean_len_G1_allother_cycles'], axis=1)
    
            dfs.append(df_len_G1_nomean)
    
    
            # if pos == 'Position_30':
                # print (df[df['Cell_ID'] == 12])
            # Compute lengths of S first cycle for each cell
            df_S = df[df['cell_cycle_stage'] == 'S']
            # if pos == 'Position_30':
            #     print (df_S[df_S['Cell_ID'] == 12])
            df_S_firstcycle = df_S[df_S['generation_num'] == 1.0]
            # if pos == 'Position_30':
            #     print (df_S_firstcycle)
    
    
            grouped_S_firstcycle = df_S_firstcycle.groupby('Cell_ID')
    
            df_len_S = grouped_S_firstcycle.size().to_frame('len_S_cycle1')
    
            # if pos == 'Position_30':
            #
            #     print (df_len_S)
    
            bud_emerg_size_df =[]
            df_S_firstcycle.columns = df_S_firstcycle.columns.str.replace("'"," ")
            for cell in df_S_firstcycle['Cell_ID'].unique():
                cell_df_S_firstcycle = df_S_firstcycle[df_S_firstcycle['Cell_ID']== cell]
                first_S_frame = cell_df_S_firstcycle['frame_i'].iloc[0]
                last_S_frame = cell_df_S_firstcycle['frame_i'].iloc[-1]
                bud_ID = cell_df_S_firstcycle['relative_ID'].min()
    
    
    
                bud_df = df[df['Cell_ID']== bud_ID]
    
                bud_df_0 = bud_df [bud_df['generation_num'] == 0]
                bud_df_last = bud_df[bud_df['frame_i'] == last_S_frame]
    
                if len(bud_df_0) != 0:
                    bud_size_emerg = (bud_df_0['cell_vol_fl']).iloc[0]
                if len(bud_df_last) != 0:
                    bud_size_div = (bud_df_last['cell_vol_fl']).iloc[0]
    
                mother_size_emerg = (cell_df_S_firstcycle['cell_vol_fl'].iloc[0])
                mother_size_emerg = (cell_df_S_firstcycle['cell_vol_fl'].iloc[0])
                mother_size_div = (cell_df_S_firstcycle['cell_vol_fl'].iloc[-1])
    
                sys_size_emerg = bud_size_emerg + mother_size_emerg
                sys_size_div = bud_size_div + mother_size_div
                vol_added_S_bud = bud_size_div - bud_size_emerg
                vol_added_S_mother = mother_size_div - mother_size_emerg
                vol_added_S_tot = sys_size_div - sys_size_emerg
    
                cycle= 1
                bud_emerg_list = [pos, cycle, cell, mother_size_emerg, mother_size_div, vol_added_S_mother,
                                    bud_size_emerg, bud_size_div, vol_added_S_bud,
                                    sys_size_emerg, sys_size_div, vol_added_S_tot]
                bud_emerg_size_df.append(bud_emerg_list)
            bud_emerg_size_df = pd.DataFrame(bud_emerg_size_df, columns=['Position','cell_cycle_num','Cell_ID',
                                                                            'mother_size_emerg', 'mother_size_div', 'vol_added_S_mother',
                                                                             'bud_size_emerg', 'bud_size_div', 'vol_added_S_bud',
                                                                             'sys_size_emerg', 'sys_size_div', 'vol_added_S_tot'])
    
    
    
    
            # Get the last frame_i of S for each group.
            # If the last frame_i is equal to the last overall frame_i then that cell didn't finish S
            df_last_frame_i_S = (
                                  grouped_S_firstcycle['frame_i']
                                  .last()
                                  .to_frame('last_frame_i_S_cycle1')
                                  )
    
            df_last_frame_i_S = df_last_frame_i_S[df_last_frame_i_S['last_frame_i_S_cycle1'] == last_frame_i]
    
            # Put NaN on cells that did not finish S
            IDs_not_finished_S_cycle1 = df_last_frame_i_S.index
    
            df_len_S.loc[IDs_not_finished_S_cycle1] = np.nan
    
    
    
            # Compute lengths of S cycles > 1 for each cell
            cycles = df_S['generation_num'].unique()
            cycles_greater_than1 = [cycle for cycle in cycles if cycle > 1.0]
            dfs_S_cycles_greater_than1 = []
            colnames = []
    
    
            # Iterate all cycles > 1 and calculate length of each one of them
            bud_emerg_size_df_remaining_cyc =[]
            for cycle in cycles_greater_than1:
                df_S_cycle = df_S[df_S['generation_num'] == cycle]
                grouped_S_cycle = df_S_cycle.groupby('Cell_ID')
                colname = f'len_S_cycle{cycle}'
                colnames.append(colname)
                df_len_S_cycle = grouped_S_cycle.size().to_frame(colname)
                # Get the last frame_i of S for each group.
                df_last_frame_i_S = (
                                      grouped_S_cycle['frame_i']
                                      .last()
                                      .to_frame('last_frame_i')
                                      )
    
                df_last_frame_i_S = df_last_frame_i_S[df_last_frame_i_S['last_frame_i'] == last_frame_i]
    
                # Put NaN on cells that did not finish S
                IDs_not_finished_S_cycle = df_last_frame_i_S.index
    
                df_len_S_cycle.loc[IDs_not_finished_S_cycle] = np.nan
    
                dfs_S_cycles_greater_than1.append(df_len_S_cycle)
    
                bud_emerg_size_df_ccr =[]
                df_S_cycle.columns = df_S_cycle.columns.str.replace("'"," ")
                for cell in df_S_cycle['Cell_ID'].unique():
                    cell_df_S_ccr = df_S_cycle[df_S_cycle['Cell_ID']== cell]
                    bud_ID_ccr = cell_df_S_ccr['relative_ID'].min()
                    first_S_frame = cell_df_S_ccr['frame_i'].iloc[0]
                    last_S_frame = cell_df_S_ccr['frame_i'].iloc[-1]
    
                    bud_df_ccr = df[df['Cell_ID']==bud_ID_ccr]
                    bud_df_0_ccr = bud_df_ccr[bud_df_ccr['generation_num'] == 0]
                    bud_size_emerg = (bud_df_0_ccr['cell_vol_fl']).iloc[0]
                    if (last_S_frame in (bud_df_ccr['frame_i'].unique())) == True:
                        bud_df_last = bud_df_ccr[bud_df_ccr['frame_i'] == last_S_frame]
    
                        bud_size_div = bud_df_last['cell_vol_fl'].iloc[0]
                    else:
    
                        bud_size_div = np.nan
    
                    mother_size_emerg = (cell_df_S_ccr['cell_vol_fl'].iloc[0])
                    mother_size_div = (cell_df_S_ccr['cell_vol_fl'].iloc[-1])
                    sys_size_emerg = bud_size_emerg + mother_size_emerg
                    sys_size_div = bud_size_div + mother_size_div
    
    
                    vol_added_S_bud = bud_size_div - bud_size_emerg
                    vol_added_S_mother = mother_size_div - mother_size_emerg
                    vol_added_S_tot = sys_size_div - sys_size_emerg
    
    
    
                    bud_emerg_list_ccr = [pos, cycle, cell, mother_size_emerg, mother_size_div, vol_added_S_mother,
                                                            bud_size_emerg, bud_size_div, vol_added_S_bud,
                                                            sys_size_emerg, sys_size_div, vol_added_S_tot]
                    bud_emerg_size_df_ccr.append(bud_emerg_list_ccr)
                bud_emerg_size_df_ccr = pd.DataFrame(bud_emerg_size_df_ccr, columns=['Position','cell_cycle_num','Cell_ID',
                                                                                    'mother_size_emerg', 'mother_size_div', 'vol_added_S_mother',
                                                                                    'bud_size_emerg','bud_size_div', 'vol_added_S_bud',
                                                                                    'sys_size_emerg', 'sys_size_div','vol_added_S_tot'])
                if len (bud_emerg_size_df_ccr) != 0:
                    bud_emerg_size_df_remaining_cyc.append(bud_emerg_size_df_ccr)
            if len(bud_emerg_size_df_remaining_cyc) != 0:
                bud_emerg_size_df_remaining_cyc = pd.concat(bud_emerg_size_df_remaining_cyc, ignore_index=True)
    
                bud_emerg_size_df_allcyc = bud_emerg_size_df.merge(bud_emerg_size_df_remaining_cyc, on = ['Position','cell_cycle_num','Cell_ID',
                   'mother_size_emerg', 'mother_size_div', 'vol_added_S_mother',
                    'bud_size_emerg','bud_size_div', 'vol_added_S_bud',
                    'sys_size_emerg', 'sys_size_div','vol_added_S_tot'], how ='outer')
    
            else:
                bud_emerg_size_df_allcyc = bud_emerg_size_df
    
    
    
            s_size.append(bud_emerg_size_df_allcyc)
    
    
            # Join df_len_S with all other cycles to create one dataframe
            df_len_S = df_len_S.join(dfs_S_cycles_greater_than1, how='outer')
    
    
    
            # Compute the mean length of all other cycles
            df_len_S['mean_len_S_allother_cycles'] = df_len_S[colnames].mean(axis=1)
    
    
    
            # Put nan for cells in the first frame at the first cell cycle
            #print ( df['date'].iloc[0])
    
    
            df_mothers_frame0 = df[(df['frame_i']==0) & (df['generation_num']== 2.0)]
    
            IDs_frame0 = df_mothers_frame0['Cell_ID'].unique()
    
            if (date == 3012023) & (pos == "Position_3"):
                display(df_len_S)
            # print (pos, IDs_frame0)
            # for it, lenind in zip(IDs_frame0, df_len_S.index) :
            #     if it == lenind:
            # if pos == 'Position_30':
            #     print (df_len_S)
            #         print (it, lenind)
            for frame0moms in IDs_frame0:
                # print (frame0moms in df_len_S.index)
                if (frame0moms in df_len_S.index) == True:
                    df_len_S['len_S_cycle2'].loc[frame0moms] = np.nan
                    df_len_S['len_S_cycle1'].loc[frame0moms] = np.nan
    
    
            df_len_S_nomean = df_len_S.drop(['mean_len_S_allother_cycles'], axis=1)
    
            dfss.append(df_len_S_nomean)
    
    
        cellsize_birth_cc1 = pd.DataFrame(cellsize, columns =colz)
        cellsize_div_ccr = pd.DataFrame(cellsize_ccr, columns = colzy)
    
    
        cellsize = cellsize_birth_cc1.merge(cellsize_div_ccr, how = 'outer')
        # print (cellsize)
    
        final_s_size_df = pd.concat(s_size, ignore_index=True)
        # print ("hello")
        # print (final_s_size_df)
    
    
    
        # # cols= ['Position','Mean G1 Length in Cell Cycle 1', 'Number of G1 phases counted (cc1)','Mean S Length in Cell Cycle 1','Number of S phases counted (cc1)', 'Mean G1 Length in Remaining Cell Cycles (Weighted)', 'Number of G1 phases counted', 'Mean S Length in Remaining Cell Cycles', 'Number of S phases counted']
        # ExperimentSummary = pd.DataFrame(expsumm, columns = cols)
        #
        # ExperimentSummary.to_csv(r'C:\Users\yagya.chadha\Desktop\Yagya\Python_MyScripts\ExperimentSummary.csv')
        # print (ExperimentSummary)
    
        big_df_G1_len = pd.concat(dfs, keys=keys)
    
        big_df_G1_len.rename(columns={ "len_G1_cycle1" : "1", "len_G1_cycle2" : "2",
                                        "len_G1_cycle3" : "3", "len_G1_cycle4" : "4",
                                        "len_G1_cycle5" : "5", "len_G1_cycle6" : "6",
                                        "len_G1_cycle7" : "7", "len_G1_cycle8" : "8"}, inplace=True)
    
    
        big_df_S_len = pd.concat(dfss, keys=keys)
    
        big_df_S_len.rename(columns={ "len_S_cycle1" : "1", "len_S_cycle2" : "2",
                                        "len_S_cycle3" : "3", "len_S_cycle4" : "4",
                                        "len_S_cycle5" : "5", "len_S_cycle6" : "6",
                                        "len_S_cycle7" : "7", "len_S_cycle8" : "8"}, inplace=True)
    
        listy = []
        for col in big_df_G1_len.columns:
            listy.append(big_df_G1_len[col].to_frame('len_G1'))
    
        big_df_G1_len_cycles = pd.concat(listy, keys=big_df_G1_len.columns,
                                   names=['cell_cycle_num','Position_n', 'Cell_ID']
                                   ).reset_index()
        big_df_G1_len_cycles['date'] = date
    
    
    
        listy1 = []
        for col1 in big_df_S_len.columns:
            listy1.append(big_df_S_len[col1].to_frame('len_S'))
    
        big_df_S_len_cycles = pd.concat(listy1, keys=big_df_S_len.columns,
                                   names=['cell_cycle_num','Position_n', 'Cell_ID']
                                   ).reset_index()
        big_df_S_len_cycles['date'] = date
    
    
        if date == 20012021:
            big_df_G1_len_cycles['Strain'] = ["MMY" if ((posy == "Position_1") or
                                                        (posy == "Position_2") or
                                                        (posy == "Position_3") or
                                                        (posy == "Position_4") or
                                                        (posy == "Position_5") or
                                                        (posy == "Position_6"))
                                                        else
                                                        "BCK2DEL" if (((posy == "Position_7") or
                                                        (posy == "Position_8") or
                                                        (posy == "Position_9") or
                                                        (posy == "Position_10") or
                                                        (posy == "Position_11")))
    
                                                        else
                                                        "WHI5DEL" if (((posy == "Position_12") or
                                                        (posy == "Position_13") or
                                                        (posy == "Position_14") or
                                                        (posy == "Position_15") or
                                                        (posy == "Position_16") or
                                                        (posy == "Position_17") or
                                                        (posy == "Position_18") or
                                                        (posy == "Position_19")))
    
                                                        else
                                                        "WHI5_BCK2_DD" if ((posy == "Position_20") or
                                                        (posy == "Position_21") or
                                                        (posy == "Position_22") or
                                                        (posy == "Position_23") or
                                                        (posy == "Position_24") or
                                                        (posy == "Position_25") or
                                                        (posy == "Position_26") or
                                                        (posy == "Position_27") or
                                                        (posy == "Position_28"))
    
                                                        else
                                                        "WHI5_CCR4_DD" if ((posy == "Position_29") or
                                                        (posy == "Position_30") or
                                                        (posy == "Position_31") or
                                                        (posy == "Position_32") or
                                                        (posy == "Position_33") or
                                                        (posy == "Position_34") or
                                                        (posy == "Position_35") or
                                                        (posy == "Position_36") or
                                                        (posy == "Position_37"))
    
    
    
                                                        else "unknown"
                                                        for posy in (big_df_G1_len_cycles['Position_n'])]
            # print (big_df_G1_len_cycles)
    
            big_df_S_len_cycles['Strain'] = ["MMY" if ((posy == "Position_1") or
                                                        (posy == "Position_2") or
                                                        (posy == "Position_3") or
                                                        (posy == "Position_4") or
                                                        (posy == "Position_5") or
                                                        (posy == "Position_6"))
                                                        else
                                                        "BCK2DEL" if (((posy == "Position_7") or
                                                        (posy == "Position_8") or
                                                        (posy == "Position_9") or
                                                        (posy == "Position_10") or
                                                        (posy == "Position_11")))
    
                                                        else
                                                        "WHI5DEL" if (((posy == "Position_12") or
                                                        (posy == "Position_13") or
                                                        (posy == "Position_14") or
                                                        (posy == "Position_15") or
                                                        (posy == "Position_16") or
                                                        (posy == "Position_17") or
                                                        (posy == "Position_18") or
                                                        (posy == "Position_19")))
    
                                                        else
                                                        "WHI5_BCK2_DD" if ((posy == "Position_20") or
                                                        (posy == "Position_21") or
                                                        (posy == "Position_22") or
                                                        (posy == "Position_23") or
                                                        (posy == "Position_24") or
                                                        (posy == "Position_25") or
                                                        (posy == "Position_26") or
                                                        (posy == "Position_27") or
                                                        (posy == "Position_28"))
    
                                                        else
                                                        "WHI5_CCR4_DD" if ((posy == "Position_29") or
                                                        (posy == "Position_30") or
                                                        (posy == "Position_31") or
                                                        (posy == "Position_32") or
                                                        (posy == "Position_33") or
                                                        (posy == "Position_34") or
                                                        (posy == "Position_35") or
                                                        (posy == "Position_36") or
                                                        (posy == "Position_37"))
    
    
    
                                                        else "unknown"
                                                        for posy in (big_df_S_len_cycles['Position_n'])]
            # print (big_df_S_len_cycles)
    
        elif date == 21012021:
            big_df_G1_len_cycles['Strain'] = ["MMY" if ((posy == "Position_1") or
                                                            (posy == "Position_2") or
                                                            (posy == "Position_3") or
                                                            (posy == "Position_4") or
                                                            (posy == "Position_5") or
                                                            (posy == "Position_6"))
                                                            else
                                                            "BCK2DEL" if (((posy == "Position_7") or
                                                            (posy == "Position_8") or
                                                            (posy == "Position_9") or
                                                            (posy == "Position_10") or
                                                            (posy == "Position_11")))
                                                            else
                                                            "CCR4DEL" if ((posy == "Position_12") or
                                                            (posy == "Position_13") or
                                                            (posy == "Position_14") or
                                                            (posy == "Position_15"))
                                                            else
                                                            "WHI5DEL" if ((posy == "Position_16") or
                                                            (posy == "Position_17") or
                                                            (posy == "Position_18") or
                                                            (posy == "Position_19") or
                                                            (posy == "Position_20") or
                                                            (posy == "Position_21") or
                                                            (posy == "Position_22"))
                                                            else
                                                            "WHI5_BCK2_DD" if ((posy == "Position_23") or
                                                            (posy == "Position_24") or
                                                            (posy == "Position_25") or
                                                            (posy == "Position_26") or
                                                            (posy == "Position_27"))
                                                              
                                                            else
                                                            "WHI5_CCR4_DD" if ((posy == "Position_28") or
                                                            (posy == "Position_29") or
                                                            (posy == "Position_30") or
                                                            (posy == "Position_31") or
                                                            (posy == "Position_32") or
                                                            (posy == "Position_33"))
                                                            
                                                            else "unknown"
                                                            for posy in (big_df_G1_len_cycles['Position_n'])]
    
            # print (big_df_G1_len_cycles)
    
            big_df_S_len_cycles['Strain'] = ["MMY" if ((posy == "Position_1") or
                                                            (posy == "Position_2") or
                                                            (posy == "Position_3") or
                                                            (posy == "Position_4") or
                                                            (posy == "Position_5") or
                                                            (posy == "Position_6"))
                                                            else
                                                            "BCK2DEL" if (((posy == "Position_7") or
                                                            (posy == "Position_8") or
                                                            (posy == "Position_9") or
                                                            (posy == "Position_10") or
                                                            (posy == "Position_11")))
                                                            else
                                                            "CCR4DEL" if ((posy == "Position_12") or
                                                            (posy == "Position_13") or
                                                            (posy == "Position_14") or
                                                            (posy == "Position_15"))
                                                            else
                                                            "WHI5DEL" if ((posy == "Position_16") or
                                                            (posy == "Position_17") or
                                                            (posy == "Position_18") or
                                                            (posy == "Position_19") or
                                                            (posy == "Position_20") or
                                                            (posy == "Position_21") or
                                                            (posy == "Position_22"))
                                                            else
                                                            "WHI5_BCK2_DD" if ((posy == "Position_23") or
                                                            (posy == "Position_24") or
                                                            (posy == "Position_25") or
                                                            (posy == "Position_26") or
                                                            (posy == "Position_27"))
                                                              
                                                            else
                                                            "WHI5_CCR4_DD" if ((posy == "Position_28") or
                                                            (posy == "Position_29") or
                                                            (posy == "Position_30") or
                                                            (posy == "Position_31") or
                                                            (posy == "Position_32") or
                                                            (posy == "Position_33"))
                                                            
                                                            else "unknown"
                                                            for posy in (big_df_S_len_cycles['Position_n'])]
    
            # print (big_df_S_len_cycles)
    
        #print (big_df_G1_len_cycles.dtypes)
        #print (cellsize.dtypes)
    
        big_df_G1_len_cycles['cell_cycle_num']=pd.to_numeric(big_df_G1_len_cycles['cell_cycle_num'])
        #print (big_df_G1_len_cycles.dtypes)
        big_df_size_and_G1_len_cycles = pd.merge(big_df_G1_len_cycles, cellsize, on = ['Position_n','Cell_ID','cell_cycle_num'], how ='outer')
        # print (big_df_size_and_G1_len_cycles)
    
        big_df_S_len_cycles['cell_cycle_num']=pd.to_numeric(big_df_S_len_cycles['cell_cycle_num'])
        #print (big_df_G1_len_cycles.dtypes)
        final_s_size_df = final_s_size_df.rename(columns = {'Position': 'Position_n'})
    
        big_df_size_and_S_len_cycles = pd.merge(big_df_S_len_cycles, final_s_size_df, on = ['Position_n','Cell_ID','cell_cycle_num'], how ='outer')
    
        # print (big_df_size_and_S_len_cycles)
    
    
        for j in big_df_size_and_G1_len_cycles.index:
            if (np.isnan((big_df_size_and_G1_len_cycles['len_G1']).iloc[j])) == True:
                (big_df_size_and_G1_len_cycles['Birth vol fl'].iloc[j]) = np.nan
                (big_df_size_and_G1_len_cycles['Div vol fl'].iloc[j]) = np.nan
                (big_df_size_and_G1_len_cycles['vol_added_G1'].iloc[j]) = np.nan
                (big_df_size_and_G1_len_cycles['Birth frame'].iloc[j]) = np.nan
                (big_df_size_and_G1_len_cycles['last_G1_frame'].iloc[j]) = np.nan
                (big_df_size_and_G1_len_cycles['size_G1_end'].iloc[j]) = np.nan
                (big_df_size_and_G1_len_cycles['Div frame'].iloc[j]) = np.nan
        #print (big_df_size_and_G1_len_cycles)
    
        # print("yes here",(big_df_size_and_G1_len_cycles['Birth vol fl']).max())
    
    
        for h in big_df_size_and_S_len_cycles.index:
            if (np.isnan((big_df_size_and_S_len_cycles['len_S']).iloc[h])) == True:
                (big_df_size_and_S_len_cycles['mother_size_emerg'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['mother_size_div'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['vol_added_S_mother'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['bud_size_emerg'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['bud_size_div'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['vol_added_S_bud'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['sys_size_emerg'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['sys_size_div'].iloc[h]) = np.nan
                (big_df_size_and_S_len_cycles['vol_added_S_tot'].iloc[h]) = np.nan
    
        #print (big_df_size_and_S_len_cycles)
    
        date_df.append(big_df_size_and_G1_len_cycles)
        date_df.append(big_df_size_and_S_len_cycles)
    

# print (date_df)
finaldate_df = pd.concat(date_df).reset_index()
display(finaldate_df)
finaldate_df['total vol added'] = np.nan




[0 1]
[0 1]


,index,cell_cycle_num,Position_n,Cell_ID,len_G1,date,Strain,Birth frame,Birth vol fl,last_G1_frame,size_G1_end,vol_added_G1,Div frame,Div vol fl,len_S,mother_size_emerg,mother_size_div,vol_added_S_mother,bud_size_emerg,bud_size_div,vol_added_S_bud,sys_size_emerg,sys_size_div,vol_added_S_tot
0,0,1,Position_1,1,13.0,20012021,MMY,7.0,67.5,19.0,86.848,19.348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,Position_1,1,8.0,20012021,MMY,NaN,NaN,52.0,99.053,4.998,45.0,94.055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,3,Position_1,1,7.0,20012021,MMY,NaN,NaN,79.0,91.238,-3.785,73.0,95.023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,4,Position_1,1,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,5,Position_1,1,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24758,3891,4,Position_9,170,NaN,21012021,BCK2DEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24759,3892,5,Position_9,170,NaN,21012021,BCK2DEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24760,3893,6,Position_9,170,NaN,21012021,BCK2DEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24761,3894,7,Position_9,170,NaN,21012021,BCK2DEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import warnings

with warnings.catch_warnings(record=True):
    for date in finaldate_df['date'].unique():
        date_df  = finaldate_df[finaldate_df['date'] == date]
        for pos in date_df['Position_n'].unique():
            pos_df = date_df[date_df['Position_n'] == pos]
            for cell in pos_df['Cell_ID'].unique():
                cell_df = pos_df[pos_df['Cell_ID'] == cell]
                for cycle in cell_df['cell_cycle_num'].unique():
                    cycle_df =  cell_df[cell_df['cell_cycle_num'] == cycle]
                    # cycle_df['vol_added_G1']= cycle_df['vol_added_G1'].fillna(0)
                    # cycle_df['vol_added_S_tot']= cycle_df['vol_added_S_tot'].fillna(0)
    
    
    
                    if (len (cycle_df['total vol added']) > 1):
    
                        total_vol_added = (cycle_df['vol_added_G1'].iloc[0]) + (cycle_df['vol_added_S_tot'].iloc[1])
                        ind = finaldate_df[(finaldate_df['date'] == date) & (finaldate_df['Position_n'] == pos) & (finaldate_df['Cell_ID'] == cell) & (finaldate_df['cell_cycle_num'] == cycle)].index
    
                        finaldate_df['total vol added'].iloc[ind[0]] = total_vol_added
                        finaldate_df['total vol added'].iloc[ind[1]] = total_vol_added
    
    




finaldate_df.to_csv(r"Y:\test_dfs_for_final_code\CombinedTRs SCD LCM analysed.csv")

